# Chapter 2: Reading & Writing Data

Pandas can read from and write to a wide variety of formats.  
In this chapter we cover:

1. `read_csv` / `to_csv` (the most common)
2. `read_excel` / `to_excel`
3. `read_json` / `to_json`
4. `read_clipboard`
5. Key parameters you should know
6. Reading from URLs
7. Corner cases: encoding, mixed dtypes, large files

In [ ]:
import pandas as pd
import numpy as np
import os

## 2.1 Create Sample Data Files

In [ ]:
# Create a sample CSV to work with
sample_data = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'age': [25, 30, 35, 28, 32],
    'city': ['New York', 'London', 'Paris', 'Tokyo', 'Sydney'],
    'salary': [70000, 80000, 90000, 75000, 85000],
    'start_date': ['2020-01-15', '2019-06-01', '2021-03-20', '2020-11-10', '2018-08-05'],
})

# Write CSV
sample_data.to_csv('sample_employees.csv', index=False)

# Write JSON
sample_data.to_json('sample_employees.json', orient='records', indent=2)

print('Sample files created!')
sample_data

## 2.2 `read_csv` — The Workhorse

`pd.read_csv()` is the function you will use most often.

In [ ]:
# Basic read
df = pd.read_csv('sample_employees.csv')
df

In [ ]:
# Useful parameters
df = pd.read_csv(
    'sample_employees.csv',
    index_col='name',           # use 'name' as index
    parse_dates=['start_date'],  # auto-parse dates
    dtype={'salary': float},     # explicit dtype
)
df.info()

### Common `read_csv` Parameters

| Parameter | Purpose | Example |
|-----------|---------|--------|
| `sep` | Delimiter | `sep='\t'` for TSV |
| `header` | Row number for column names | `header=None` if no header |
| `index_col` | Column to use as row index | `index_col=0` |
| `usecols` | Read only certain columns | `usecols=['name', 'age']` |
| `nrows` | Read only first N rows | `nrows=100` |
| `dtype` | Force column types | `dtype={'zip': str}` |
| `na_values` | Additional NA markers | `na_values=['N/A', '?']` |
| `parse_dates` | Parse as datetime | `parse_dates=['date_col']` |
| `encoding` | File encoding | `encoding='utf-8'` |
| `chunksize` | Read in chunks | `chunksize=10000` |

In [ ]:
# Using usecols and nrows
df_subset = pd.read_csv(
    'sample_employees.csv',
    usecols=['name', 'salary'],
    nrows=3
)
df_subset

## 2.3 `to_csv` — Writing CSV Files

In [ ]:
# Write without index
df.to_csv('output_no_index.csv', index=False)

# Write with custom separator
df.reset_index().to_csv('output_tsv.tsv', sep='\t', index=False)

# Write only certain columns
df.reset_index().to_csv('output_partial.csv', columns=['name', 'age'], index=False)

print('CSV files written!')

## 2.4 Reading & Writing JSON

In [ ]:
# Read JSON
df_json = pd.read_json('sample_employees.json')
df_json

In [ ]:
# Write JSON (different orientations)
print('--- records ---')
print(df_json.head(2).to_json(orient='records', indent=2))

print('\n--- columns ---')
print(df_json.head(2).to_json(orient='columns', indent=2))

## 2.5 Reading & Writing Excel

Requires `openpyxl` (for `.xlsx`) or `xlrd` (for `.xls`).

```python
# Read
df = pd.read_excel('file.xlsx', sheet_name='Sheet1')

# Write
df.to_excel('output.xlsx', sheet_name='Results', index=False)

# Multiple sheets
with pd.ExcelWriter('multi_sheet.xlsx') as writer:
    df1.to_excel(writer, sheet_name='Data', index=False)
    df2.to_excel(writer, sheet_name='Summary', index=False)
```

## 2.6 `read_clipboard`

Copy a table from Excel / a website and run:

```python
df = pd.read_clipboard()  # reads tab-separated data from clipboard
```

Extremely handy for quick ad-hoc analysis!

## 2.7 Reading from URLs

In [ ]:
# Pandas can read directly from a URL
# Example (using a well-known public dataset):
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv'

# Uncomment the line below to actually fetch from the URL:
# df_iris = pd.read_csv(url)
# df_iris.head()

print('Any function that takes a filepath can also take a URL!')

## 2.8 Corner Cases & Large Files

### Encoding Issues

In [ ]:
# When you get UnicodeDecodeError, try:
# df = pd.read_csv('file.csv', encoding='latin-1')
# df = pd.read_csv('file.csv', encoding='cp1252')

# Tip: use 'errors' in Python's open() or chardet library to detect encoding
print('Common encodings: utf-8, latin-1, cp1252, iso-8859-1')

### Mixed Data Types

In [ ]:
# If a column has mixed types, Pandas defaults to 'object'
mixed_data = pd.DataFrame({
    'id': [1, 2, 3, 4, 5],
    'value': ['10', '20', 'N/A', '40', 'fifty']  # mixed!
})

# Convert with errors='coerce' — non-numeric become NaN
mixed_data['value_clean'] = pd.to_numeric(mixed_data['value'], errors='coerce')
mixed_data

### Large Files with `chunksize`

In [ ]:
# For files too large to fit in memory, read in chunks
# Each chunk is a DataFrame — process incrementally

# Simulate with our small file
chunk_iter = pd.read_csv('sample_employees.csv', chunksize=2)

total_salary = 0
for i, chunk in enumerate(chunk_iter):
    total_salary += chunk['salary'].sum()
    print(f'Chunk {i}: {len(chunk)} rows, running salary total = {total_salary}')

print(f'\nFinal total salary: {total_salary}')

## 🧹 Cleanup Temp Files

In [ ]:
import glob
for f in glob.glob('output_*.csv') + glob.glob('output_*.tsv'):
    os.remove(f)
    print(f'Removed {f}')
print('Cleanup done!')

## 📝 Key Takeaways

| Topic | Key Point |
|-------|----------|
| **read_csv** | Most common; learn its parameters well |
| **parse_dates** | Always parse date columns on read |
| **usecols** | Read only what you need |
| **chunksize** | Process large files incrementally |
| **encoding** | Try `latin-1` or `cp1252` if `utf-8` fails |
| **to_csv** | Use `index=False` unless you need the index |